# Indic Synthetic-Speech Pipeline — Colab (T4) stage-by-stage runbook

Validate **one stage at a time** on the GPU, then commit and move on.

**The loop (per stage):** edit locally in VSCode → `git push` → here run `!git pull` → run the stage → inspect the manifest + listen to audio → fix & repeat → commit `"stage N validated"`.

**Why outputs go to Drive:** each stage reads the previous stage's manifest, and Colab runtimes disconnect. With `out_dir` on Drive (see `config.colab.yaml`) the manifests + audio survive restarts, so the per-stage sessions below can run in *separate* runtimes.

**transformers conflict:** IndicF5 pins `==4.49.0` but Gemma-3 needs `>=4.50`. We handle it by giving each session its own install and **restarting the runtime** between Sessions 1→2→3.

> ⚠️ Never hardcode an HF token in a committed cell. We read it via `getpass`. If you ever leaked one, revoke it at https://huggingface.co/settings/tokens .

## 0. One-time setup (run at the start of every session)
Mount Drive, clone-or-pull the repo, set the HF token. Set `REPO_URL` to your GitHub repo.

In [6]:
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only
os.makedirs('/content/drive/MyDrive/indic_synth/out', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/synthetic-data-pipeline
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.31 KiB | 670.00 KiB/s, done.
From https://github.com/rahulkolayikkath/synthetic-data-pipeline
   c41c6e2..7b56e84  main       -> origin/main
Updating c41c6e2..7b56e84
Fast-forward
 requirements.txt                            |  2 ++
 src/indic_synth/quality_control/config.py   |  3 ++-
 src/indic_synth/quality_control/pipeline.py | 10 +++++++++-
 3 files changed, 13 insertions(+), 2 deletions(-)


In [7]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token: ')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


---
## Session 1 — §4.2 acquisition + §4.3 audio engineering
No transformers model; CPU is fine. Run §4.2, inspect, then §4.3, inspect.

In [4]:
!pip install -q pyarrow pandas fsspec huggingface_hub datasets soundfile soxr librosa pyyaml
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [5]:
# §4.2 — pull the gender-balanced reference voice bank from Kathbath
!python scripts/run.py --config config.colab.yaml --stages data_acquisition

14:29:50 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
14:29:50 INFO    run | =========== stage: data_acquisition ===========
14:29:50 INFO    run | Acquisition config: {'source': 'hf', 'repo_id': 'ai4bharat/Kathbath', 'local_dir': 'fake_kathbath', 'languages': ['hindi', 'malayalam'], 'split': 'valid', 'speakers_per_language': 10, 'clips_per_speaker': 4, 'min_total_speakers': 20, 'gender_balance': True, 'ref_min_dur': 3.0, 'ref_max_dur': 15.0, 'seed': 1234, 'out_dir': '/content/drive/MyDrive/indic_synth/out', 'hf_token': True, 'max_retries': 4, 'retry_backoff': 2.0, 'force_catalog': False}
14:29:51 INFO    run | [hindi] cataloging 2 parquet file(s) (audio column skipped)
14:30:03 INFO    run | [hindi] valid-00000-of-00002.parquet -> 1576 rows
14:30:15 INFO    run | [hindi] valid-00001-of-00002.parquet -> 1575 rows
14:30:15 INFO    run | [malayalam] cataloging 2 parquet file(s) (audio column skipped)
14:30:22 INFO    run | [malayalam] valid-0000

In [8]:
# inspect + listen to a reference clip
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/reference_manifest.jsonl --n 3
import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/ref_audio/*'))[:2]:
    print(w); display(Audio(w))


=== /content/drive/MyDrive/indic_synth/out/reference_manifest.jsonl ===
rows: 80

-- categorical --
  status           {'downloaded': 80}
  lang             {'hindi': 40, 'malayalam': 40}
  gender           {'male': 40, 'female': 40}

-- numeric (min / mean / max) --
  duration         3.135 / 7.443 / 13.816

distinct (lang, speaker): 20

-- 3 sample rows --
  {"ref_id": "hindi_spk934_778", "lang": "hindi", "speaker_id": 934, "gender": "male", "source_repo": "ai4bharat/Kathbath", "source_split": "valid", "row_index": 778, "fname": "844424933473550-934-m.m4a", "ref_text": "नाइट्रोजन यौगीकीकरण जैसे द्वारा मिट्टी में प्राकृतिक रूप से किया जाता है", "duration": 8.336, "local_audio_path": "ref_audio/hindi_spk934_778.flac", "status": "downloaded"}
  {"ref_id": "hindi_spk934_563", "lang": "hindi", "speaker_id": 934, "gender": "male", "source_repo": "ai4bharat/Kathbath", "source_split": "valid", "row_index": 563, "fname": "844424933541613-934-m.m4a", "ref_text": "काच शलाका एवं नली का निर्माण 

/content/drive/MyDrive/indic_synth/out/ref_audio/hindi_spk1179_1287.flac


In [9]:
# §4.3 — decode / resample to 24 kHz mono / normalize
!python scripts/run.py --config config.colab.yaml --stages audio_engineering
!python scripts/inspect_manifest.py $OUT/prepared_manifest.jsonl --n 3

14:39:05 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
14:39:05 INFO    run | =========== stage: audio_engineering ===========
14:39:05 INFO    run | 80 downloaded clips, 0 already prepared, 80 to process
14:39:07 INFO    run | Prepare complete: {"stage": "audio_engineering", "elapsed_sec": 2.44, "target_sr": 24000, "norm": "peak", "trim": false, "prepared": 80, "failed": 0, "sr_in": {"16000": 80}, "flags": {"resampled_up": 80, "input_clipped": 9}, "backends": {"soundfile": 80}, "prepared_manifest": "/content/drive/MyDrive/indic_synth/out/prepared_manifest.jsonl"}
14:39:07 INFO    run | Pipeline done in 2.5s. Final dataset: /content/drive/MyDrive/indic_synth/out/dataset_manifest.jsonl

=== /content/drive/MyDrive/indic_synth/out/prepared_manifest.jsonl ===
rows: 80

-- categorical --
  status           {'prepared': 80}
  lang             {'hindi': 40, 'malayalam': 40}
  gender           {'male': 40, 'female': 40}
  resample_method  {'soxr_hq': 8

In [10]:
# verify a prepared clip really is 24 kHz mono, and listen
import soundfile as sf, glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/prepared_audio/*.wav'))[:2]:
    i = sf.info(w); print(w, i.samplerate, 'Hz', i.channels, 'ch'); display(Audio(w))

/content/drive/MyDrive/indic_synth/out/prepared_audio/hindi_spk1179_1075.wav 24000 Hz 1 ch


/content/drive/MyDrive/indic_synth/out/prepared_audio/hindi_spk1179_1287.wav 24000 Hz 1 ch


**✅ If §4.2/§4.3 look right** (20 speakers, gender-balanced, 24 kHz mono refs): commit `"stage 4.2/4.3 validated"` from VSCode.

**Watch-outs:** gated 403 → accept Kathbath terms on the hub. If no files are found, the real HF layout / language-folder names may differ from the `datasets/ai4bharat/Kathbath/<lang>/valid-*.parquet` glob in `src/indic_synth/data_acquisition/hf_io.py` — adjust `languages` in `config.colab.yaml` or the glob. Kathbath clips may be **m4a** (decoded via librosa/ffmpeg in `audio_engineering/prepare.py`).

---
## Session 2 — §4.4 sentence generation (Gemma-3)
**Restart the runtime first** (Runtime → Restart), then re-run section 0, then this.
Needs `transformers>=4.50` + a GPU.

In [6]:
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/indic_synth/hf_cache"

In [3]:
!pip install -q "transformers>=4.50" accelerate bitsandbytes sentence-transformers fasttext-wheel indic-num2words pyyaml
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 32.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [7]:
!pip uninstall -y fasttext fasttext-wheel
!pip install -q fasttext-numpy2-wheel

Found existing installation: fasttext-wheel 0.9.2
Uninstalling fasttext-wheel-0.9.2:
  Successfully uninstalled fasttext-wheel-0.9.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 53.7 MB/s eta 0:00:0000:0100:01


In [12]:
# in Colab, after you push from local
!git pull && pip install -e . -q

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 298 bytes | 298.00 KiB/s, done.
From https://github.com/rahulkolayikkath/synthetic-data-pipeline
   2220446..6216893  main       -> origin/main
Updating 2220446..6216893
Fast-forward
 config.colab.yaml | 3 ++-
 1 file changed, 2 insertions(+), 1 deletion(-)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [13]:
# §4.4 — grid-balanced, validated Indic sentences (checkpointed; safe to re-run)
!python scripts/run.py --config config.colab.yaml --stages sentence_generation

16:49:13 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
16:49:13 INFO    run | =========== stage: sentence_generation ===========
Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Fetching 5 files:  20% 1/5 [00:00<00:00,  4.93it/s]
Fetching 5 files: 100% 5/5 [00:00<00:00, 15.46it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights:   0% 1/1065 [00:07<2:16:54,  7.72s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 1065/1065 [04:52<00:00,  3.64it/s] 
16:54:35 INFO    run | Model loaded: google/gemma-3-12b-it
Loading weights: 100% 199/199 [00:08<00:00, 23.02it/s]
16:55:48 INFO    run | Resumed 1040 sentences from checkpoint.
16:55:48 INFO    run | Grid: 1 cells x quota 13 (~13 target sentences); already have

In [14]:
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/sentences.jsonl --n 5
import json
print(open(f'{OUT}/sentences_summary.json').read())


=== /content/drive/MyDrive/indic_synth/out/sentences.jsonl ===
rows: 1053

-- categorical --
  language         {'hi': 533, 'ml': 520}
  sentence_type    {'declarative': 273, 'interrogative': 260, 'imperative': 260, 'exclamatory': 260}
  topic            {'Daily Commute': 104, 'Local Cuisine': 104, 'Tech Troubleshooting': 104, 'Weather Reports': 104, 'Health and Wellness': 104, 'Shopping and Markets': 104, 'Festivals and Culture': 104, 'Travel and Tourism': 104, 'Education and School': 104, 'Banking and Money': 104, 'Sports': 13}

-- numeric (min / mean / max) --
  word_count       3.000 / 5.483 / 13.000
  char_count       14.000 / 36.665 / 998.000

-- 5 sample rows --
  {"id": "hi_000000", "language": "hi", "language_name": "Hindi", "topic": "Daily Commute", "sentence_type": "declarative", "text": "मैं रोज़ सुबह सात बजे घर से निकलता हूँ।", "word_count": 9, "char_count": 38, "seed": 2720962256}
  {"id": "hi_000001", "language": "hi", "language_name": "Hindi", "topic": "Daily Commute", 

**✅ Check:** per-language / per-type / per-topic counts are balanced, QC yield is reasonable, and sampled sentences read fluently. Commit `"stage 4.4 validated"`.

**Watch-outs:** accept the **Gemma-3 license**; `bitsandbytes` 4-bit needs the GPU; `lid.176.bin` (~126 MB) downloads into `out_dir`; confirm `processor.apply_chat_template(...)` works for the installed transformers (`sentence_generation/models.py`).

---
## Session 3 — §4.5 TTS (IndicF5) + §4.6 QC
**Restart the runtime first**, re-run section 0, then this. IndicF5 from source + `transformers==4.49.0`.

In [15]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml
!pip install -q -e .

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 65.5 MB/s eta 0:00:00:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.0/113.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 k

In [16]:
# §4.5 — IndicF5 speaks each sentence in a same-language speaker's voice
!python scripts/run.py --config config.colab.yaml --stages tts_generation
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/tts_manifest.jsonl --n 3

17:00:53 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
17:00:53 INFO    run | =========== stage: tts_generation ===========
config.json: 100% 350/350 [00:00<00:00, 2.21MB/s]
model.py: 5.77kB [00:00, 21.6MB/s]
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
A new version of the following files was downloaded from https://huggingface.co/ai4bharat/IndicF5:
-

In [4]:
# listen to a few synthesized utterances
OUT = '/content/drive/MyDrive/indic_synth/out'
import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/tts_audio/*.wav'))[:4]:
    print(w); display(Audio(w))

/content/drive/MyDrive/indic_synth/out/tts_audio/hi_000000.wav


/content/drive/MyDrive/indic_synth/out/tts_audio/hi_000001.wav


/content/drive/MyDrive/indic_synth/out/tts_audio/hi_000002.wav


/content/drive/MyDrive/indic_synth/out/tts_audio/hi_000003.wav


In [8]:
%cd /content/synthetic-data-pipeline
!pip install -r requirements.txt
!pip install -e .

/content/synthetic-data-pipeline
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 86.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 61.5 MB/s eta 0:00:00
Obtaining file:///content/synthetic-data-pipeline
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done
  Created wheel for indic-synth: filename=indic_synth-0.1.0-0.editable-

In [9]:
!pip install onnxruntime   # silences the warnings; not required

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.3 MB/s eta 0:00:0000:01:00:01


In [10]:
# §4.6 — three QC gates -> final dataset_manifest.jsonl
!python scripts/run.py --config config.colab.yaml --stages quality_control
!python scripts/inspect_manifest.py $OUT/dataset_manifest.jsonl --n 3
import json
print(open(f'{OUT}/qc_summary.json').read())

20:02:55 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out seed=1234
20:02:55 INFO    run | =========== stage: quality_control ===========
20:02:55 INFO    run | 866 utterances, 12 already QC'd, 854 to check
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Fetching 404 files: 100% 404/404 [00:23<00:00, 17.41it/s]
Download complete: 100% 2.56G/2.56G [00:23<00:00, 46.2MB/s]                Please check FRAME_DURATION_MS. The timestamps can be inaccurate
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(

hyperparams.yaml: 100% 1.92k/1.92k [00:00<00:00, 3.54MB/s]

embedding_model.ckpt:   0% 0.00/83.3M [00:00<?, ?B/s]
embedding_model.ckpt: 100% 83.3M/83.3M [00:01<0

In [13]:
import json
import os
from typing import Dict, Iterable, Iterator, List
def read_jsonl(path: str) -> List[Dict]:
    """Read a JSONL file into a list of dicts. Missing file -> empty list."""
    if not os.path.exists(path):
        return []
    rows: List[Dict] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [17]:
# sanity-check thresholds: listen to a couple of QC failures
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][20:40]
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))

pass: 610 / 866
hi_000034 CER= 0.0 spk= 0.6883701086044312 ['truncation(tail/peak=0.31 (>0.25 => suspect cut-off))']


hi_000035 CER= 0.0 spk= 0.5684078335762024 ['truncation(tail/peak=0.46 (>0.25 => suspect cut-off))', 'speaker_cos(cos=0.568 (min 0.6))']


hi_000040 CER= 0.0 spk= 0.7747061848640442 ['looping_audio(max_voiced_repeat=0.90 @lag=0.71s (loop if >0.85))']


hi_000041 CER= 0.0 spk= 0.7112143635749817 ['truncation(tail/peak=0.35 (>0.25 => suspect cut-off))']


hi_000048 CER= 0.0 spk= 0.7461082339286804 ['truncation(tail/peak=0.26 (>0.25 => suspect cut-off))']


...

hi_000087 CER= 0.0 spk= 0.5766266584396362 ['speaker_cos(cos=0.577 (min 0.6))']


hi_000089 CER= 0.2222222222222222 spk= 0.7543778419494629 ['cer(CER=0.222 (max 0.15))']


hi_000096 CER= 0.125 spk= 0.7997717261314392 ['truncation(tail/peak=0.29 (>0.25 => suspect cut-off))']


hi_000099 CER= 0.05405405405405406 spk= 0.7082749009132385 ['truncation(tail/peak=0.34 (>0.25 => suspect cut-off))']


hi_000100 CER= 0.0 spk= 0.514637291431427 ['speaker_cos(cos=0.515 (min 0.6))']


**✅ Done** when `dataset_manifest.jsonl` holds ~1000 validated utterances across 2 languages / 20 speakers with a sensible QC pass rate. Commit `"stage 4.5/4.6 validated"`.

**Watch-outs:** IndicF5 git build + `trust_remote_code`; confirm the `tts_model(text, ref_audio_path=..., ref_text=...)` call + int16→float32 handling in `tts_generation/models.py`; T4 VRAM for IndicF5. For QC: the `indic-conformer` `model(wav, lang_code, 'ctc')` signature and the **lang_code it expects** (map `hi`/`ml` if needed); speechbrain ECAPA load. If conformer needs a different transformers than 4.49, run §4.6 in its own session — the Drive manifests make that safe.